In [14]:
import os
os.makedirs('/content/drive/MyDrive/models', exist_ok=True)

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install scikit-learn mlflow -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 114.3 MB/s eta 0:00:0000:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 118.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/879.5 kB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [11]:
import shutil, os, numpy as np

# Correct path — files are in MyDrive/models/
drive_data = '/content/drive/MyDrive/models/'
local_data = 'data/processed/'
os.makedirs(local_data, exist_ok=True)

for f in ['X_train.npy','X_val.npy','X_test.npy',
          'y_train.npy','y_val.npy','y_test.npy']:
    shutil.copy(drive_data + f, local_data + f)
    print(f"Copied {f}")

# Verify
X_train = np.load('data/processed/X_train.npy')
y_train = np.load('data/processed/y_train.npy')
print(f"\nX_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"Labels: {np.unique(y_train, return_counts=True)}")
import tensorflow as tf
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

Copied X_train.npy
Copied X_val.npy
Copied X_test.npy
Copied y_train.npy
Copied y_val.npy
Copied y_test.npy

X_train: (490, 10, 15)
y_train: (490,)
Labels: (array([0, 1]), array([209, 281]))
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [12]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error, classification_report
import os

def load_data():
    X_train = np.load('data/processed/X_train.npy')
    X_test  = np.load('data/processed/X_test.npy')
    X_val   = np.load('data/processed/X_val.npy')
    y_train = np.load('data/processed/y_train.npy')
    y_test  = np.load('data/processed/y_test.npy')
    y_val   = np.load('data/processed/y_val.npy')

    # Binarize if not already binary
    if len(set(y_train)) > 2:
        median = np.median(y_train)
        y_train = (y_train > median).astype(int)
        y_test  = (y_test  > median).astype(int)
        y_val   = (y_val   > median).astype(int)

    print(f"Data loaded — X_train: {X_train.shape}, X_test: {X_test.shape}")
    return X_train, X_test, X_val, y_train, y_test, y_val

def build_rnn(input_shape, units=64, dropout_rate=0.2):
    model = Sequential([
        SimpleRNN(units, activation='tanh', return_sequences=False,
                  input_shape=input_shape),
        BatchNormalization(),
        Dropout(dropout_rate),
        Dense(32, activation='relu'),
        Dropout(0.1),
        Dense(1, activation='sigmoid')
    ], name='RNN_Model')

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    model.summary()
    return model

def train_rnn(units=64, dropout_rate=0.2, epochs=50, batch_size=32):
    X_train, X_test, X_val, y_train, y_test, y_val = load_data()

    model = build_rnn(
        input_shape=(X_train.shape[1], X_train.shape[2]),
        units=units,
        dropout_rate=dropout_rate
    )

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=7,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=3, verbose=1)
    ]

    history = model.fit(
        X_train, y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_data=(X_val, y_val),
        callbacks=callbacks,
        verbose=1
    )

    y_pred = (model.predict(X_test) > 0.5).astype(int).flatten()
    acc  = accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred, average='weighted')
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    print(f"\nRNN Results — Accuracy: {acc:.4f} | F1: {f1:.4f} | RMSE: {rmse:.4f}")
    print(classification_report(y_test, y_pred))

    os.makedirs('models', exist_ok=True)
    model.save('models/best_rnn.keras')
    print("RNN model saved to models/best_rnn.keras")

    return model, history, acc, f1, rmse


if __name__ == '__main__':
    model, history, acc, f1, rmse = train_rnn(units=64, dropout_rate=0.2)

import shutil, os
drive_models = '/content/drive/MyDrive/models/'
os.makedirs(drive_models, exist_ok=True)
os.makedirs('models', exist_ok=True)
model.save('models/best_rnn.keras')
shutil.copy('models/best_rnn.keras', drive_models + 'best_rnn.keras')
print("RNN saved to Drive.")

Data loaded — X_train: (490, 10, 15), X_test: (105, 10, 15)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "RNN_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_1 (SimpleRNN)        │ (None, 64)             │         5,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,489 (29.25 KB)

 Trainable params: 7,361 (28.75 KB)

 Non-trainable params: 128 (512.00 B)

Epoch 1/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 7s 195ms/step - accuracy: 0.4898 - loss: 0.7895 - val_accuracy: 0.4000 - val_loss: 0.7174 - learning_rate: 0.0010
Epoch 2/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5204 - loss: 0.7559 - val_accuracy: 0.4000 - val_loss: 0.7024 - learning_rate: 0.0010
Epoch 3/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5408 - loss: 0.7335 - val_accuracy: 0.6095 - val_loss: 0.6897 - learning_rate: 0.0010
Epoch 4/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5224 - loss: 0.7104 - val_accuracy: 0.6000 - val_loss: 0.6811 - learning_rate: 0.0010
Epoch 5/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5408 - loss: 0.7299 - val_accuracy: 0.6000 - val_loss: 0.6760 - learning_rate: 0.0010
Epoch 6/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5551 - loss: 0.7009 - val_accuracy: 0.6000 - val_loss: 0.6727 - learning_rate: 0.0010
Epoch 7/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4918 - loss: 0.7403 - val_accuracy

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [13]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error, classification_report
import os

def load_data():
    X_train = np.load('data/processed/X_train.npy')
    X_test  = np.load('data/processed/X_test.npy')
    X_val   = np.load('data/processed/X_val.npy')
    y_train = np.load('data/processed/y_train.npy')
    y_test  = np.load('data/processed/y_test.npy')
    y_val   = np.load('data/processed/y_val.npy')

    if len(set(y_train)) > 2:
        median = np.median(y_train)
        y_train = (y_train > median).astype(int)
        y_test  = (y_test  > median).astype(int)
        y_val   = (y_val   > median).astype(int)

    print(f"Data loaded — X_train: {X_train.shape}, X_test: {X_test.shape}")
    return X_train, X_test, X_val, y_train, y_test, y_val

def build_lstm(input_shape, units=64, dropout_rate=0.2):
    model = Sequential([
        LSTM(units, activation='tanh', return_sequences=True,
             input_shape=input_shape),
        BatchNormalization(),
        Dropout(dropout_rate),
        LSTM(units // 2, activation='tanh', return_sequences=False),
        BatchNormalization(),
        Dropout(dropout_rate),
        Dense(32, activation='relu'),
        Dropout(0.1),
        Dense(1, activation='sigmoid')
    ], name='LSTM_Model')

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    model.summary()
    return model

def train_lstm(units=64, dropout_rate=0.2, epochs=50, batch_size=32):
    X_train, X_test, X_val, y_train, y_test, y_val = load_data()

    model = build_lstm(
        input_shape=(X_train.shape[1], X_train.shape[2]),
        units=units,
        dropout_rate=dropout_rate
    )

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=7,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=3, verbose=1)
    ]

    history = model.fit(
        X_train, y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_data=(X_val, y_val),
        callbacks=callbacks,
        verbose=1
    )

    y_pred = (model.predict(X_test) > 0.5).astype(int).flatten()
    acc  = accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred, average='weighted')
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    print(f"\nLSTM Results — Accuracy: {acc:.4f} | F1: {f1:.4f} | RMSE: {rmse:.4f}")
    print(classification_report(y_test, y_pred))

    os.makedirs('models', exist_ok=True)
    model.save('models/best_lstm.keras')
    print("LSTM model saved to models/best_lstm.keras")

    return model, history, acc, f1, rmse

if __name__ == '__main__':
    model, history, acc, f1, rmse = train_lstm(units=64, dropout_rate=0.2)

import shutil, os
drive_models = '/content/drive/MyDrive/models/'
os.makedirs(drive_models, exist_ok=True)
os.makedirs('models', exist_ok=True)
model.save('models/best_lstm.keras')
shutil.copy('models/best_lstm.keras', drive_models + 'best_lstm.keras')
print("LSTM saved to Drive.")

Data loaded — X_train: (490, 10, 15), X_test: (105, 10, 15)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "LSTM_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 10, 64)         │        20,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 10, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 10, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 34,369 (134.25 KB)

 Trainable params: 34,177 (133.50 KB)

 Non-trainable params: 192 (768.00 B)

Epoch 1/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - accuracy: 0.5286 - loss: 0.7610 - val_accuracy: 0.4000 - val_loss: 0.6999 - learning_rate: 0.0010
Epoch 2/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5306 - loss: 0.7154 - val_accuracy: 0.5810 - val_loss: 0.6920 - learning_rate: 0.0010
Epoch 3/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5143 - loss: 0.7212 - val_accuracy: 0.6000 - val_loss: 0.6884 - learning_rate: 0.0010
Epoch 4/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5388 - loss: 0.7221 - val_accuracy: 0.4000 - val_loss: 0.7011 - learning_rate: 0.0010
Epoch 5/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5184 - loss: 0.7246 - val_accuracy: 0.4000 - val_loss: 0.6963 - learning_rate: 0.0010
Epoch 6/50
13/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5728 - loss: 0.6798
Epoch 6: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5429 - loss: 0.7029 - val_accur

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [14]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error, classification_report
import os

def load_data():
    X_train = np.load('data/processed/X_train.npy')
    X_test  = np.load('data/processed/X_test.npy')
    X_val   = np.load('data/processed/X_val.npy')
    y_train = np.load('data/processed/y_train.npy')
    y_test  = np.load('data/processed/y_test.npy')
    y_val   = np.load('data/processed/y_val.npy')

    if len(set(y_train)) > 2:
        median = np.median(y_train)
        y_train = (y_train > median).astype(int)
        y_test  = (y_test  > median).astype(int)
        y_val   = (y_val   > median).astype(int)

    print(f"Data loaded — X_train: {X_train.shape}, X_test: {X_test.shape}")
    return X_train, X_test, X_val, y_train, y_test, y_val

def build_gru(input_shape, units=64, dropout_rate=0.2):
    model = Sequential([
        GRU(units, activation='tanh', return_sequences=False,
            input_shape=input_shape),
        BatchNormalization(),
        Dropout(dropout_rate),
        Dense(64, activation='relu'),
        Dropout(dropout_rate),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ], name='GRU_Model')

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    model.summary()
    return model

def train_gru(units=64, dropout_rate=0.2, epochs=50, batch_size=32):
    X_train, X_test, X_val, y_train, y_test, y_val = load_data()

    model = build_gru(
        input_shape=(X_train.shape[1], X_train.shape[2]),
        units=units,
        dropout_rate=dropout_rate
    )

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=7,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=3, verbose=1)
    ]

    history = model.fit(
        X_train, y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_data=(X_val, y_val),
        callbacks=callbacks,
        verbose=1
    )

    y_pred = (model.predict(X_test) > 0.5).astype(int).flatten()
    acc  = accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred, average='weighted')
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    print(f"\nGRU Results — Accuracy: {acc:.4f} | F1: {f1:.4f} | RMSE: {rmse:.4f}")
    print(classification_report(y_test, y_pred))

    os.makedirs('models', exist_ok=True)
    model.save('models/best_gru.keras')
    print("GRU model saved to models/best_gru.keras")

    return model, history, acc, f1, rmse

if __name__ == '__main__':
    model, history, acc, f1, rmse = train_gru(units=64, dropout_rate=0.2)


import shutil
drive_models = '/content/drive/MyDrive/models/'

model_gru, history_gru, acc_gru, f1_gru, rmse_gru = train_gru(units=64, dropout_rate=0.2)
shutil.copy('models/best_gru.keras', drive_models + 'best_gru.keras')
print("GRU saved to Drive")

Data loaded — X_train: (490, 10, 15), X_test: (105, 10, 15)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "GRU_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_2 (GRU)                     │ (None, 64)             │        15,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,081 (86.25 KB)

...

 Total params: 22,081 (86.25 KB)

 Trainable params: 21,953 (85.75 KB)

 Non-trainable params: 128 (512.00 B)

Epoch 1/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 3s 42ms/step - accuracy: 0.5306 - loss: 0.7299 - val_accuracy: 0.6000 - val_loss: 0.6822 - learning_rate: 0.0010
Epoch 2/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.5388 - loss: 0.7151 - val_accuracy: 0.6000 - val_loss: 0.6824 - learning_rate: 0.0010
Epoch 3/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.5694 - loss: 0.6999 - val_accuracy: 0.6000 - val_loss: 0.6859 - learning_rate: 0.0010
Epoch 4/50
12/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5501 - loss: 0.6908
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5571 - loss: 0.6902 - val_accuracy: 0.6000 - val_loss: 0.6832 - learning_rate: 0.0010
Epoch 5/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.5735 - loss: 0.6874 - val_accuracy: 0.6000 - val_loss: 0.6787 - learning_rate: 5.0000e-04
Epoch 6/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.5449 - loss: 0.7021 - val_

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
